# Chapter 10 — Many labels: flexibility, regularisation, censoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/the-cannon-tutorial/blob/main/notebooks/10_many_labels.ipynb)

**Goal.** Build a model with twelve labels — $T_{\rm eff}$, $\log g$, [Fe/H] and nine abundances [X/Fe] — and add
to The Cannon of Chapter 2 the tools that published codes use for many labels (Casey et al. 2016, C16): models of
higher polynomial order, L1 regularisation, censoring, several starting points for the test step, and a test of
whether a star lies inside the training set. For each tool, measure on APOGEE spectra what it buys.

**You will learn**
1. that more labels and a more flexible model both make the stellar parameters more precise, and to tell the two
   apart;
2. what L1 regularisation does (sparse coefficients, better predicted spectra) and when it improves the labels:
   only when the training set is not much larger than the number of coefficients;
3. that censoring — letting each abundance act only near its own lines — changes the precision element by
   element (Holtzman et al. 2018);
4. why a convex-hull test of "inside the training set" flags almost every star in many dimensions.

**The new code** is in the block `more.py` below (tested in `tests/test_cannon.py`):
- `vectorize_n(x, order)` and `dvectorize_n`: label vectors of any polynomial order (for order 2 identical to
  `vectorize`);
- `fit_theta_l1`: at every pixel, minimise $\chi^2 + \Lambda \sum_{d \ge 1} |\theta_d|$ (C16, eq. 2 and 5), with
  some coefficients forced to zero by a censoring mask. For $\Lambda = 0$ it is the weighted least squares of
  Chapter 2; for $\Lambda > 0$ it uses ADMM (Boyd et al. 2011), which handles the strongly correlated quadratic
  terms much better than coordinate descent;
- `train_more`: the training step of Chapter 2 (coefficients and maximum-likelihood scatter, iterated) with these
  options; `test_more`: the test step from several starting points; `censor_mask`; `in_hull`.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are extracts of APOGEE DR17 and Gaia DR3, published on Zenodo (https://doi.org/10.5281/zenodo.23813051).
# MANIFEST.json lists, for each file, the query that made it and its SHA-256 checksum.
DATA_URL = 'https://zenodo.org/records/23813051/files/{}?download=1'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL.format('MANIFEST.json')).read())


def load(name, cache=Path('data')):
    """Download <name> (.parquet table or .npz arrays) from Zenodo if needed, check its SHA-256, read it.

    A .parquet file is returned as a pandas DataFrame, an .npz file as a dict of numpy arrays.
    """
    ext = '.npz' if name.startswith(('spectra_', 'rvs_')) else '.parquet'
    path = cache / f'{name}{ext}'
    sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    if not path.exists() or sha(path) != MANIFEST[name]['sha256']:   # first use, or a stale/damaged copy
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        part = path.with_suffix('.part')
        urllib.request.urlretrieve(DATA_URL.format(path.name), part)   # download to a temporary file first
        if sha(part) != MANIFEST[name]['sha256']:
            raise IOError(f'{path.name}: checksum does not match the manifest (incomplete download?)')
        part.replace(path)                                         # only a verified file takes the final name
    # the checksum guarantees that everyone runs this chapter on exactly the same data
    if ext == '.parquet':
        return pd.read_parquet(path)
    with np.load(path, allow_pickle=False) as f:   # allow_pickle=False: arrays only, no code can be loaded
        return {k: f[k] for k in f.files}
# --- Spectra and continuum normalisation (as in Chapters 0 and 3) ----------------------------------------
NPIX = 8575
wave = 10**(4.179 + 6e-6 * np.arange(NPIX))               # apStar vacuum wavelength grid [Angstrom]
CHIPS = [(15150, 15800), (15865, 16425), (16480, 16950)]  # the three detectors [A], inside the edges of Chapter 0
BAD_BITS = sum(1 << b for b in [0, 1, 2, 3, 4, 5, 6, 7, 12])   # pixels given zero weight (Chapter 0)


def prepare(sp, rows=slice(None)):
    """flux and err (float64) of the spectra file sp; err = inf where there are no data, a BAD_BITS flag, or
    outside CHIPS (the detector edges, where only some spectra have data)."""
    flux = sp['flux'][rows].astype(float)
    err = sp['err'][rows].astype(float)
    on_chip = np.any([(wave > lo) & (wave < hi) for lo, hi in CHIPS], axis=0)   # pixels at the detector edges and
    bad = ((sp['mask'][rows] & BAD_BITS) > 0) | ~np.isfinite(flux) | ~np.isfinite(err) | (err <= 0) | ~on_chip  # gaps: no weight
    return np.where(bad, 0.0, flux), np.where(bad, np.inf, err)


def cheb_normalise(flux, err, cont, order=2):
    """Divide by a Chebyshev polynomial of the given order fitted, per detector, to the continuum pixels `cont`
    (boolean, NPIX) with weights 1/err^2 (N15 sect. 2.3 and 5.3). Returns normalised flux and error;
    pixels outside the detectors get err = inf."""
    fn, en = np.zeros_like(flux), np.full_like(err, np.inf)
    for lo, hi in CHIPS:
        c = (wave > lo) & (wave < hi)
        t = (wave[c] - (lo + hi) / 2) / ((hi - lo) / 2)          # wavelength mapped to [-1, 1]
        B = np.polynomial.chebyshev.chebvander(t, order)         # (pixels, order+1) basis
        use = cont[c]
        w = 1 / err[:, c][:, use]**2                             # (N, continuum pixels); 0 for flagged pixels
        # one small weighted least-squares problem per star: (B^T W B) a = B^T W f
        BtWB = np.einsum('pi,np,pj->nij', B[use], w, B[use])
        BtWf = np.einsum('pi,np->ni', B[use], w * flux[:, c][:, use])
        a = np.linalg.solve(BtWB, BtWf[:, :, None])[:, :, 0]     # (N, order+1); b as a stack of column vectors
        cont_fit = a @ B.T                                       # (N, pixels of this detector)
        fn[:, c] = flux[:, c] / cont_fit
        en[:, c] = err[:, c] / cont_fit
    return fn, en
# --- The Cannon (as written in Chapter 2) -----------------------------------------------------------------
from scipy.optimize import least_squares


def vectorize(x):
    """Quadratic label vector v(x) = [1, x_k, x_k x_l (k <= l)] for scaled labels x of shape (N, K): (N, D)."""
    x = np.atleast_2d(x)
    k, l = np.triu_indices(x.shape[1])
    return np.hstack([np.ones((len(x), 1)), x, x[:, k] * x[:, l]])


def dvectorize(x):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    K = len(x)
    k, l = np.triu_indices(K)
    J = np.zeros((1 + K + len(k), K))
    J[1:1 + K] = np.eye(K)
    for i, (a, b) in enumerate(zip(k, l)):
        J[1 + K + i, a] += x[b]
        J[1 + K + i, b] += x[a]
    return J


def fit_theta(V, flux, var):
    """Weighted least squares at every pixel: theta (P, D) and its covariance (P, D, D); weights 1/var (N, P).
    Zero-weight entries (var = inf) drop out of the sums."""
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)                       # NaN flux at zero weight must not poison the sums
    N, Dv = V.shape
    A = ((V[:, :, None] * V[:, None, :]).reshape(N, Dv * Dv).T @ w).T.reshape(-1, Dv, Dv)
    b = (V.T @ (w * flux)).T
    cov = np.linalg.pinv(A)                                 # pinv: pixels without data give theta = 0, not an error
    return np.einsum('pde,pe->pd', cov, b), cov


S_GRID = np.r_[0, np.logspace(-4, 0, 41)]                  # 0, then 1e-4 ... 1 in steps of 0.1 dex


def fit_scatter(resid, err2):
    """Maximum-likelihood scatter s per pixel for fixed residuals: best value on the grid S_GRID, refined by a
    parabola in log s (Chapter 2). Zero-weight entries (err2 = inf) are ignored."""
    use = np.isfinite(err2) & np.isfinite(resid)
    r2 = np.where(use, resid**2, 0.0)
    e2 = np.where(use, err2, 1.0)
    nll = np.array([np.sum(use * (r2 / (e2 + s**2) + np.log(e2 + s**2)), axis=0) for s in S_GRID])
    best = np.argmin(nll, axis=0)
    j = np.clip(best, 2, len(S_GRID) - 2)
    p = np.arange(nll.shape[1])
    a, b, c = nll[j - 1, p], nll[j, p], nll[j + 1, p]
    shift = np.clip(0.5 * (a - c) / np.where(a - 2 * b + c > 0, a - 2 * b + c, np.inf), -1, 1)   # stay near the grid optimum
    return np.where(best == 0, 0.0, S_GRID[j] * np.exp(shift * np.log(S_GRID[2] / S_GRID[1])))


def train(x, flux, err, n_iter=3):
    """Training step. x: scaled labels (N, K); flux, err: (N, P) (err = inf for pixels to ignore).
    Returns theta (P, D), its covariance (P, D, D) and the scatter s (P,)."""
    V, err2 = vectorize(x), err**2
    s = np.zeros(flux.shape[1])
    for _ in range(n_iter):
        theta, cov = fit_theta(V, flux, err2 + s**2)
        s = fit_scatter(flux - V @ theta.T, err2)
    theta, cov = fit_theta(V, flux, err2 + s**2)
    return theta, cov, s


def test(flux, err, theta, s, x0=None):
    """Test step for one spectrum: scaled labels (K,), covariance (K, K), chi^2, number of pixels used, converged?"""
    K = int(round((np.sqrt(8 * theta.shape[1] + 1) - 3) / 2))   # invert D = 1 + K + K(K+1)/2
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize(x)[0]) * w
    jac = lambda x: -(th @ dvectorize(x)) * w[:, None]
    r = least_squares(res, np.zeros(K) if x0 is None else x0, jac=jac, method='lm')
    # covariance from the Jacobian at the optimum (pinv: no crash if a label is unconstrained); r.success: converged
    return r.x, np.linalg.pinv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum(), r.success


def train_clean(x, flux, err, cut=3.0):
    """Train, drop the training stars whose mean squared residual per pixel (in units of sigma^2 + s^2) exceeds
    `cut`, and train again (Chapter 4). Returns theta, its covariance, s, and the boolean mask of stars kept."""
    theta, cov, s = train(x, flux, err)
    z2 = (flux - vectorize(x) @ theta.T)**2 / (err**2 + s**2)
    z2 = np.where(np.isfinite(err) & (theta[:, 0] != 0), z2, np.nan)
    keep = np.nanmean(z2, axis=1) < cut
    theta, cov, s = train(x[keep], flux[keep], err[keep])
    return theta, cov, s, keep


def scatter_stats(d):
    """Summary of differences d (e.g. Cannon - reference): mean (bias), standard deviation, root-mean-square error,
    robust standard deviation 1.4826 x median absolute deviation (= std for a Gaussian, insensitive to outliers),
    and the fraction of points more than 3 robust std from the median (the tails)."""
    d = np.asarray(d)[np.isfinite(d)]
    med = np.median(d)
    rob = 1.4826 * np.median(np.abs(d - med))
    return {'bias': d.mean(), 'std': d.std(), 'rmse': np.sqrt(np.mean(d**2)), 'robust': rob,
            'tail': np.mean(np.abs(d - med) > 3 * rob)}
# --- More of The Cannon (Chapter 10): any polynomial order, censoring, L1 regularisation, several starting points,
# --- and a test of whether labels lie inside the training set -------------------------------------------------
from itertools import combinations_with_replacement
import warnings
from scipy.optimize import linprog


def label_terms(K, order=2):
    """The terms of the label vector after the leading 1: tuples of label indices, e.g. (0,), (0, 1), (2, 2, 2)."""
    return [t for p in range(1, order + 1) for t in combinations_with_replacement(range(K), p)]


def vectorize_n(x, order=2):
    """Label vector of polynomial order `order` for scaled labels x (N, K): (N, D). For order 2 it equals vectorize."""
    x = np.atleast_2d(x)
    return np.hstack([np.ones((len(x), 1))] + [np.prod(x[:, list(t)], axis=1, keepdims=True)
                                                for t in label_terms(x.shape[1], order)])


def dvectorize_n(x, order=2):
    """Derivative dv/dx of the label vector of one star (x of shape (K,)): (D, K)."""
    T = label_terms(len(x), order)
    J = np.zeros((1 + len(T), len(x)))
    for i, t in enumerate(T):
        for j in range(len(t)):                              # d/dx of x_a x_b x_c ...: drop one factor at a time
            J[1 + i, t[j]] += np.prod(x[list(t[:j] + t[j + 1:])])
    return J


def censor_mask(allowed, order=2):
    """Coefficients allowed at each pixel, (P, D), from the labels allowed at each pixel, (P, K): a term may be
    non-zero only if all the labels in it are allowed. The leading coefficient (the mean spectrum) always is."""
    T = label_terms(allowed.shape[1], order)
    return np.hstack([np.ones((len(allowed), 1), bool)] + [allowed[:, list(t)].all(axis=1, keepdims=True) for t in T])


def fit_theta_l1(V, flux, var, lam=0.0, mask=None, theta0=None, chunk=2048, tol=1e-8, max_iter=20000, kappa=0.1):
    """At every pixel minimise chi^2 + lam * sum_{d >= 1} |theta_d| (Casey et al. 2016), with theta_d = 0 where
    mask (P, D) is False. lam = 0: weighted least squares, solved directly. lam > 0: ADMM (Boyd et al. 2011, §6.4) on rescaled coefficients,
    all pixels of a chunk at once, started from theta0 (P, D) if given. Weights 1/var (N, P); zero-weight entries drop
    out. Returns theta (P, D) and the number of ADMM iterations used (0 for lam = 0)."""
    N, D = V.shape
    P = flux.shape[1]
    mask = np.ones((P, D), bool) if mask is None else mask
    w = 1 / var
    flux = np.where(w > 0, flux, 0.0)
    VV = (V[:, :, None] * V[:, None, :]).reshape(N, D * D).T
    pen = np.r_[0.0, np.ones(D - 1)]                             # theta_0 (the mean spectrum) is not penalised
    theta, n_used = np.zeros((P, D)), 0
    for p0 in range(0, P, chunk):
        sl = slice(p0, p0 + chunk)
        A = (VV @ w[:, sl]).T.reshape(-1, D, D)                  # V^T W V at each pixel
        b = (V.T @ (w[:, sl] * flux[:, sl])).T                   # V^T W f
        m = mask[sl]
        if lam == 0:                                             # censored rows/columns replaced by the identity
            Am = A * (m[:, :, None] & m[:, None, :]) + np.eye(D) * ~m[:, None, :]
            theta[sl] = np.einsum('pde,pe->pd', np.linalg.pinv(Am), b * m) * m
            continue
        # Rescale each coefficient so that the curvature of chi^2 along it is 1: theta = d * y. Without this, linear and
        # quadratic terms have curvatures that differ by orders of magnitude and ADMM crawls. |theta_k| = d_k |y_k|.
        dg = np.einsum('pdd->pd', A)
        d = 1 / np.sqrt(np.where(dg > 0, 2 * dg, 1.0))
        As, bs = A * d[:, :, None] * d[:, None, :], b * d
        # ADMM: split y (quadratic part) from z (L1 part and mask), y = z; u is the scaled dual variable. The step
        # size rho scales with the soft threshold lam * d, which in these units sets how far the solution moves.
        rho = np.minimum(kappa * lam * np.median(d, axis=1), 1.0)[:, None]   # capped at the (unit) curvature
        M = np.linalg.inv(2 * As + rho[:, :, None] * np.eye(D))   # factorised once, used at every iteration
        z = np.zeros_like(b) if theta0 is None else theta0[sl] * m / d
        u = np.zeros_like(b)
        thr = pen * lam * d / rho                                # soft threshold for each scaled coefficient
        act = np.arange(len(b))                                  # pixels not yet converged
        for it in range(max_iter):
            y = np.einsum('pde,pe->pd', M[act], 2 * bs[act] + rho[act] * (z[act] - u[act]))
            v = y + u[act]
            z_new = np.where(m[act], np.sign(v) * np.maximum(np.abs(v) - thr[act], 0), 0.0)
            da = d[act]                                          # convergence judged on theta, not on y
            done = (np.abs(da * (y - z_new)).max(axis=1) < tol) & (np.abs(da * (z_new - z[act])).max(axis=1) < tol)
            z[act], u[act] = z_new, v - z_new
            act = act[~done]
            if len(act) == 0:
                break
        if len(act):
            warnings.warn(f'fit_theta_l1: {len(act)} pixels did not converge in {max_iter} iterations')
        n_used = max(n_used, it + 1)
        theta[sl] = z * d                                        # z has the exact zeros
    return theta, n_used


def train_more(x, flux, err, lam=0.0, mask=None, order=2, n_iter=3):
    """Training step with polynomial order, censoring mask and L1 regularisation; scatter by maximum likelihood as in
    `train` (Chapter 2). Returns theta (P, D) and s (P,)."""
    V, err2 = vectorize_n(x, order), err**2
    s, theta = np.zeros(flux.shape[1]), None
    for _ in range(n_iter):
        theta = fit_theta_l1(V, flux, err2 + s**2, lam, mask, theta)[0]     # each fit starts from the previous one
        s = fit_scatter(flux - V @ theta.T, err2)
    return fit_theta_l1(V, flux, err2 + s**2, lam, mask, theta)[0], s


def test_more(flux, err, theta, s, order=2, starts=None):
    """Test step for one spectrum, as `test` (Chapter 2) but for any polynomial order and from several starting
    points (rows of `starts`, scaled labels; default: the median labels, 0); of the fits that converged, the one with
    the lowest chi^2 is kept.
    Returns scaled labels, covariance, chi^2, number of pixels used, converged?"""
    K = next(k for k in range(1, 100) if 1 + len(label_terms(k, order)) == theta.shape[1])
    ok = np.isfinite(flux) & np.isfinite(err) & (theta[:, 0] != 0)
    w = 1 / np.sqrt(err[ok]**2 + s[ok]**2)
    th, f = theta[ok], flux[ok]
    res = lambda x: (f - th @ vectorize_n(x, order)[0]) * w
    jac = lambda x: -(th @ dvectorize_n(x, order)) * w[:, None]
    fits = [least_squares(res, x0, jac=jac, method='lm') for x0 in (np.zeros((1, K)) if starts is None else
                                                                     np.atleast_2d(starts))]
    r = min(fits, key=lambda r: (not r.success, np.sum(r.fun**2)))   # converged fits first, then lowest chi^2
    return r.x, np.linalg.pinv(r.jac.T @ r.jac), np.sum(r.fun**2), ok.sum(), r.success


def in_hull(x, X):
    """Is each row of x (M, K) inside the convex hull of the training labels X (N, K)? True if x is a weighted mean
    of training points with non-negative weights summing to 1: a linear-programming feasibility problem, which works
    in any number of dimensions (a Delaunay triangulation does not, beyond a few). Labels are scaled by the standard
    deviation of X so that kelvin and dex are treated alike."""
    sd = X.std(axis=0)
    A = np.vstack([(X / sd).T, np.ones(len(X))])
    status = np.array([linprog(np.zeros(len(X)), A_eq=A, b_eq=np.r_[p / sd, 1], bounds=(0, None), method='highs').status
                       for p in np.atleast_2d(x)])
    if not np.all((status == 0) | (status == 2)):              # 0: feasible (inside), 2: infeasible (outside)
        raise RuntimeError(f'linprog failed with status {set(status) - {0, 2}}')
    return status == 0
import time

labels = load('labels')
lines = load('linelist')
cont = load('continuum_pixels')['continuum'].to_numpy()
EL = ['c', 'n', 'o', 'mg', 'al', 'si', 'ca', 'mn', 'ni']
NAMES = ['teff', 'logg', 'fe_h'] + [e + '_fe' for e in EL]
TEX = [r'$T_{\rm eff}$', r'$\log g$', '[Fe/H]'] + [f'[{e.capitalize()}/Fe]' for e in EL]
data = {}
for nm in ['train', 'test']:
    lab = labels[labels['sample'] == nm].reset_index(drop=True)
    f, e = cheb_normalise(*prepare(load(f'spectra_{nm}')), cont)
    good = np.isfinite(lab[NAMES]).all(axis=1).to_numpy()   # a few stars lack Al or Mn
    data[nm] = (lab[NAMES].to_numpy()[good], f[good], e[good])
L, flux, err = data['train']
Lt, ft, et = (a[::2] for a in data['test'])                  # every second test star (S/N 50-100), to save time
val = np.random.default_rng(10).permutation(len(L)) % 5 == 0  # 20% of the training set held out for validation
tr = ~val
print(f'training {tr.sum()}, validation {val.sum()}, test {len(Lt)} stars; {len(NAMES)} labels')


def scaling(lab):
    """Pivot (median) and scale (half the 2.5-97.5 percentile range) of a set of training labels."""
    return np.median(lab, axis=0), (np.percentile(lab, 97.5, axis=0) - np.percentile(lab, 2.5, axis=0)) / 2


def label_set(f, e, theta, s, pivot, scale, order=2, starts=None):
    """Test step for every spectrum: labels in physical units and chi^2."""
    out = [test_more(f[n], e[n], theta, s, order, starts) for n in range(len(f))]
    assert all(o[4] for o in out), 'the test-step optimiser did not converge for some spectra'
    return np.array([o[0] for o in out]) * scale + pivot, np.array([o[2] for o in out])


rob = lambda d: np.array([scatter_stats(d[:, k])['robust'] for k in range(d.shape[1])])   # robust std per label
pv, sc = scaling(L[tr])
x = (L - pv) / sc

## 10.1 More labels, or more flexibility?

Chapter 8 found that a model with six labels gives better $T_{\rm eff}$ and $\log g$ than one with three: part of
the spectral effect of the abundances was being absorbed by the stellar parameters. Twelve labels push this
further — but they also make the model more flexible: 91 coefficients per pixel instead of 10. A three-label model
of *third* order has 20 coefficients and no abundances. Compare the three on the validation stars (high S/N, not
used to train the models; they did help to choose the continuum pixels in Chapter 3, which Chapter 5 found to
change the precision by less than 2%) and the test stars (S/N 50–100), with labels measured against ASPCAP's.

In [ ]:
t0 = time.time()
pv3, sc3 = scaling(L[tr, :3])
models = {'3 labels, quadratic': (3, 2), '3 labels, cubic': (3, 3), '12 labels, quadratic': (12, 2)}
fits, res = {}, {}
for name, (K, order) in models.items():
    p, c = (pv3, sc3) if K == 3 else (pv, sc)
    th, s = train_more((L[tr, :K] - p) / c, flux[tr], err[tr], order=order)
    fits[name] = (th, s)
    lab_val = label_set(flux[val], err[val], th, s, p, c, order)[0]
    lab_test = label_set(ft, et, th, s, p, c, order)[0]
    res[name] = {'validation': rob(lab_val - L[val, :K]), 'test': rob(lab_test - Lt[:, :K])}
    print(f"{name:22s} D = {th.shape[1]:3d}   validation: {res[name]['validation'][:3].round(3)}   "
          f"test: {res[name]['test'][:3].round(3)}")
th12, s12 = fits['12 labels, quadratic']
lab12_test = label_set(ft, et, th12, s12, pv, sc)[0]
print('12 labels, abundances [X/Fe], robust std (validation / test):')
print(pd.DataFrame({'validation': res['12 labels, quadratic']['validation'][3:],
                    'test': res['12 labels, quadratic']['test'][3:]}, index=NAMES[3:]).T.round(3).to_string())
print(f'{time.time() - t0:.0f} s')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for k, a in enumerate(ax):
    for j, split in enumerate(['validation', 'test']):
        a.bar(np.arange(3) + 0.38 * j - 0.19, [res[m][split][k] for m in models], 0.36,
              color=['C0', 'C1'][j], label=split + (' (high S/N)' if j == 0 else ' (S/N 50–100)'))
    a.set_xticks(range(3), [m.replace(', ', '\n') for m in models])
    a.set(ylabel=f'robust std of Cannon − ASPCAP [{["K", "dex", "dex"][k]}]', title=TEX[k])
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure 10.1.** Precision of $T_{\rm eff}$, $\log g$ and [Fe/H] (robust standard deviation of the differences from
ASPCAP) for three models trained on the same 1308 stars: three labels with the quadratic model of the earlier
chapters (10 coefficients per pixel), three labels with a cubic model (20), and twelve labels with a quadratic model
(91). Blue: the 328 validation stars, of the same high S/N as the training set; orange: 741 test stars at
S/N 50–100. On the validation stars both changes improve $T_{\rm eff}$ by a third (50 → 33–34 K): the gain from
adding labels can be had by making the three-label model more flexible, without measuring a single abundance. At
low S/N the abundances help more than the flexibility ($T_{\rm eff}$ 47 → 34 K with twelve labels, → 41 K with the
cubic model; likewise $\log g$).

**Checks.**

In [ ]:
q3, c3, q12 = (res[m]['test'] for m in models)
v3, vc3, v12 = (res[m]['validation'] for m in models)
assert vc3[0] < 0.8 * v3[0] and v12[0] < 0.8 * v3[0], 'neither more flexibility nor more labels improves T_eff'
assert q12[0] < c3[0] < q3[0] and q12[1] < c3[1] < q3[1]

## 10.2 L1 regularisation

With 91 coefficients per pixel, many of them describe effects that are not there — a dependence on [Mn/Fe] at a
pixel with no Mn line — and fitting them adds noise. C16 penalised the sum of their absolute values, which sets
many of them exactly to zero (the "lasso", Tibshirani 1996). The strength $\Lambda$ is in units of $\chi^2$ and is
chosen by validation. Following C16 (their Figs. 2–4), we scan $\Lambda$ and measure, on the validation stars:
the fraction of coefficients that are exactly zero, how well the model predicts their spectra at their ASPCAP
labels (the mean of $r^2/(\sigma^2+s^2)$ per pixel, with the scatter $s$ of the unregularised model for all), and
the precision of the labels.

In [ ]:
t0 = time.time()
LAMS = [0, 1e2, 3e2, 1e3, 3e3, 1e4]
Vval = vectorize_n(x[val])
use = th12[:, 0] != 0                                          # pixels with data
path = {}
for lam in LAMS:
    th, s = (th12, s12) if lam == 0 else train_more(x[tr], flux[tr], err[tr], lam)
    ok = np.isfinite(err[val][:, use])                        # pixels with zero weight are not counted
    z2 = np.where(ok, (flux[val][:, use] - Vval @ th[use].T)**2 / (err[val][:, use]**2 + s12[use]**2), np.nan)
    lab = label_set(flux[val], err[val], th, s, pv, sc)[0]
    path[lam] = {'zero': np.mean(th[use, 1:] == 0), 'chi2': np.nanmean(z2),
                 'rob': rob(lab - L[val])}
    print(f"Lambda = {lam:7.0f}: zero coefficients {path[lam]['zero']:.2f}, validation chi^2 per pixel "
          f"{path[lam]['chi2']:.3f}, T_eff {path[lam]['rob'][0]:.1f} K, median [X/Fe] {np.median(path[lam]['rob'][3:]):.4f}")
print(f'{time.time() - t0:.0f} s')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
lx = np.array(LAMS[1:])
ax[0].semilogx(lx, [path[l]['zero'] for l in lx], 'ko-')
ax[0].set(xlabel=r'$\Lambda$', ylabel='fraction of coefficients exactly zero', title='(a) sparsity')
ax[1].semilogx(lx, [path[l]['chi2'] for l in lx], 'ko-')
ax[1].axhline(path[0]['chi2'], color='r', ls='--', label=r'$\Lambda = 0$')
ax[1].set(xlabel=r'$\Lambda$', ylabel=r'validation $\chi^2$ per pixel at ASPCAP labels', title='(b) predicted spectra')
ax[1].legend()
ratio = np.array([path[l]['rob'] / path[0]['rob'] for l in lx])
for k in range(len(NAMES)):
    ax[2].semilogx(lx, ratio[:, k], color='C0' if k < 3 else '0.6', lw=1.5 if k < 3 else 0.8,
                   label=r'$T_{\rm eff}, \log g$, [Fe/H]' if k == 0 else ('[X/Fe]' if k == 3 else None))
ax[2].axhline(1, color='r', ls='--')
ax[2].set(xlabel=r'$\Lambda$', ylabel=r'robust std / robust std at $\Lambda = 0$', title='(c) labels')
ax[2].legend()
plt.tight_layout()
plt.show()

**Figure 10.2.** L1 regularisation of the twelve-label model trained on 1308 stars, measured on the 328 validation
stars. **(a)** The fraction of the coefficients (all but the mean spectrum, at pixels with data) that are exactly
zero rises from 12% to most of them. **(b)** Moderate regularisation predicts the spectra of stars the model has not
seen better, with a minimum near $\Lambda \approx 300$; strong regularisation predicts them worse. C16 (their
Fig. 4) found the same shape.
**(c)** But the labels do not get better: the robust standard deviation of every label, relative to the
unregularised model, stays near 1 for weak regularisation and grows for strong regularisation (blue: stellar
parameters; grey: the nine abundances). Predicting spectra and measuring labels are different tasks; with 1308
training stars for 91 coefficients, the unregularised model is not overfitting enough for the penalty to help the
labels.

### When does regularisation pay?

Overfitting grows as the number of training stars approaches the number of coefficients. Repeat the comparison
with random training sets of 120 and 300 stars (two of each, to see the scatter between random draws). The penalty
is a $\chi^2$, which grows with the number of stars, so $\Lambda$ is scaled with it: $\Lambda = 300\,N/1308$ (for
the full training set this is the $\Lambda = 300$ model above).

In [ ]:
t0 = time.time()
row = lambda n, seed, reg, r: {'N': n, 'seed': seed, 'reg': reg, 'logg': r[1], 'teff': r[0], 'x_fe': np.median(r[3:])}
size = [row(tr.sum(), 0, reg, path[300 if reg else 0]['rob']) for reg in [False, True]]   # full set: from the scan
for n in [120, 300]:
    for seed in [11, 12]:
        sub = np.flatnonzero(tr)[np.random.default_rng(seed).permutation(tr.sum())[:n]]
        p, c = scaling(L[sub])
        for lam in [0, 300 * n / tr.sum()]:
            th, s = train_more((L[sub] - p) / c, flux[sub], err[sub], lam)
            size.append(row(n, seed, lam > 0, rob(label_set(flux[val], err[val], th, s, p, c)[0] - L[val])))
size = pd.DataFrame(size).sort_values(['N', 'seed', 'reg'])
print(size.round(4).to_string(index=False))
print(f'{time.time() - t0:.0f} s')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (col, lab) in zip(ax, [('teff', r'$T_{\rm eff}$ [K]'), ('logg', r'$\log g$'), ('x_fe', 'median of the nine [X/Fe]')]):
    for reg, c, nm in [(False, 'k', r'$\Lambda = 0$'), (True, 'C3', r'$\Lambda = 300\,N/1308$')]:
        d = size[size.reg == reg]
        a.plot(d.N * (1.04 if reg else 0.96), d[col], 'o', color=c, label=nm)
        a.plot(d.groupby('N').N.first(), d.groupby('N')[col].mean(), '-', color=c, lw=1)
    a.set(xscale='log', xlabel='number of training stars', ylabel=f'robust std, {lab}')
ax[0].axhline(0, lw=0)
ax[0].legend()
for a in ax:
    a.axvline(91, color='0.6', ls=':')
ax[1].text(95, ax[1].get_ylim()[1] * 0.95, '91 coefficients', color='0.4', fontsize=8, va='top')
plt.tight_layout()
plt.show()

**Figure 10.3.** Precision on the validation stars of twelve-label models trained on random subsets of 120 and 300
stars (two subsets each) and on all 1308, without (black) and with (red) L1 regularisation; lines join the means.
The dotted line marks the number of coefficients per pixel. With 120 training stars the unregularised model
overfits — its $\log g$ is twice as bad as with 300 — and regularisation recovers much of the loss, in $\log g$ and
in the abundances. With 300 or 1308 stars this penalty no longer helps. Two caveats: there are only two random
subsets per size, and $\Lambda$ was not re-tuned for each size (a $\Lambda$ chosen separately for 300 stars might still
help a little). C16 used regularisation for a 17-label model (171 coefficients per pixel); it matters when the
training set is small for the number of labels.

One feature of this figure we have not explained: the abundances of the validation stars come out *better* from
300 training stars than from all 1308, for both random subsets and without regularisation. It is listed, with
tests to run, as an open question in `STRUCTURE.md`.

**Checks.**

In [ ]:
g = size.groupby(['N', 'reg'])[['logg', 'x_fe']].mean()
small, full = 120, tr.sum()
assert g.loc[(small, True), 'logg'] < 0.8 * g.loc[(small, False), 'logg'], 'regularisation does not help at N = 120'
assert g.loc[(small, True), 'x_fe'] < g.loc[(small, False), 'x_fe']
assert g.loc[(full, True), 'x_fe'] > 0.97 * g.loc[(full, False), 'x_fe'], 'regularisation helps at full N after all'
assert min(path[l]['chi2'] for l in LAMS[1:]) < path[0]['chi2'] and path[1e4]['chi2'] > path[0]['chi2']
assert np.median(path[1e4]['rob'] / path[0]['rob']) > 1

## 10.3 Censoring: each abundance from its own lines

Chapter 8 asked whether [Mg/Fe] is *measured* from Mg lines or *predicted* from the rest of the spectrum through
its correlations with other labels. Censoring answers by construction: a coefficient that involves [X/Fe] may be
non-zero only at pixels where element X has lines. APOGEE did this for the Cannon abundances of DR14, using the
ASPCAP abundance windows (Holtzman et al. 2018, §8), because "correlations of abundances of different elements in
the training set" would otherwise be imposed on every star labelled. We use our line list: the pixels within 1 Å
of the 40 strongest lines of the element's neutral atom, plus the CO and CN bands for carbon, CN for nitrogen, and
OH and CO for oxygen. $T_{\rm eff}$, $\log g$ and [Fe/H] act everywhere. A term like $T_{\rm eff}\times$[Mg/Fe] is
allowed only where [Mg/Fe] is.

In [ ]:
SPECIES = {'c': ['C  I', 'CO', 'CN'], 'n': ['N  I', 'CN'], 'o': ['O  I', 'OH', 'CO'], 'mg': ['MG I'], 'al': ['AL I'],
           'si': ['SI I'], 'ca': ['CA I'], 'mn': ['MN I'], 'ni': ['NI I']}


def near(species, half_width=1.0):
    """Pixels within half_width [A] of a line of the given species (vacuum wavelengths)."""
    w = lines[lines.species.isin(species)].wave_vac.to_numpy()
    return np.min(np.abs(wave[:, None] - w[None, :]), axis=1) < half_width


allowed = np.column_stack([np.ones((NPIX, 3), bool)] + [near(SPECIES[e]) for e in EL])
mask = censor_mask(allowed)
print('pixels allowed for each abundance:', dict(zip(EL, allowed[:, 3:].sum(axis=0))))
t0 = time.time()
thc, sc_c = train_more(x[tr], flux[tr], err[tr], mask=mask)
res_c = {'validation': rob(label_set(flux[val], err[val], thc, sc_c, pv, sc)[0] - L[val]),
         'test': rob(label_set(ft, et, thc, sc_c, pv, sc)[0] - Lt)}
print(pd.DataFrame({'uncensored': res['12 labels, quadratic']['test'], 'censored': res_c['test']}, index=NAMES).T.round(3).to_string())
print(f'{time.time() - t0:.0f} s')
dmg = th12[:, 1 + NAMES.index('mg_fe')] / sc[NAMES.index('mg_fe')]   # df/d[Mg/Fe] per dex, uncensored
on = th12[:, 0] != 0
out_frac = np.sum(np.abs(dmg[on & ~allowed[:, 6]])) / np.sum(np.abs(dmg[on]))
print(f'uncensored model: {out_frac:.0%} of the total |df/d[Mg/Fe]| lies at pixels more than 1 A from a Mg I line')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={'width_ratios': [1.4, 1]})
lo, hi = 15730, 15800
z = on & (wave > lo) & (wave < hi)
ax[0].fill_between(wave[z], -0.25, 0.1, where=allowed[z, 6], color='C2', alpha=0.15, lw=0, label='Mg I windows')
ax[0].plot(wave[z], th12[z, 0] - 1, 'k', lw=0.8, label=r'$\theta_0 - 1$')
ax[0].plot(wave[z], dmg[z], 'C0', lw=1.2, label=r'$\partial f/\partial$[Mg/Fe], uncensored')
ax[0].plot(wave[z], thc[z, 1 + NAMES.index('mg_fe')] / sc[NAMES.index('mg_fe')], 'C3', lw=1.2,
           label=r'$\partial f/\partial$[Mg/Fe], censored')
ax[0].set(xlim=(lo, hi), ylim=(-0.25, 0.1), xlabel='vacuum wavelength [Å]', ylabel='normalised flux (per dex)',
          title='(a) where [Mg/Fe] acts')
ax[0].legend(fontsize=8, ncol=2, loc='lower left')
for j, split in enumerate(['validation', 'test']):
    ax[1].bar(np.arange(12) + 0.38 * j - 0.19, res_c[split] / res['12 labels, quadratic'][split], 0.36,
              color=['C0', 'C1'][j], label=split)
ax[1].axhline(1, color='k', lw=0.8)
ax[1].set_xticks(range(12), TEX, rotation=60, fontsize=8)
ax[1].set(ylabel='robust std: censored / uncensored', title='(b) precision with censoring')
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure 10.4.** **(a)** The derivative of the flux with respect to [Mg/Fe] at the median labels (the linear
coefficient; away from the median the cross terms change it) in a 70 Å window, in the uncensored
(blue) and censored (red) twelve-label models; the green bands are the pixels within 1 Å of Mg I lines, the only
ones where the censored model may use [Mg/Fe]. The uncensored model also gives [Mg/Fe] small derivatives
everywhere else (the fraction printed above). **(b)** The robust standard deviation of each label with censoring,
relative to the uncensored model, for the validation (high S/N) and test (S/N 50–100) stars. At low S/N censoring
*improves* Mg, Al, Si, Ca, Mn and Ni (by 12–21%). A likely reason is that the small derivatives spread over
thousands of pixels fit noise more than signal at low S/N; we have not proved it. At high S/N censoring improves
only Mg, Al and Ca, and costs 6–17% in Si, Mn and Ni, which may draw on real information outside our windows — or on
correlations with other labels in the training set; this experiment cannot tell which. It degrades O and C at
both S/N — their information is spread over many molecular lines, which our 1 Å windows around 60 lines per molecule
capture only in part — and $T_{\rm eff}$, which loses the cross terms with the abundances away from their lines.
Holtzman et al. 2018 saw the same kind of element-by-element change for DR14 (worse Ca, still good Ni, with the
ASPCAP windows). Censoring guarantees only that an abundance acts directly on its own pixels; it still couples to
the other labels in the joint fit. All precisions here are measured against ASPCAP, which is not the truth.

**Checks.**

In [ ]:
rc = res_c['test'] / res['12 labels, quadratic']['test']
for el in ['mg_fe', 'al_fe', 'ca_fe', 'ni_fe']:
    assert rc[NAMES.index(el)] < 0.95, f'censoring does not improve {el} at low S/N'
assert rc[NAMES.index('o_fe')] > 1.1 and rc[0] > 1.1
assert out_frac > 0.5

## 10.4 Three small tools

**Several starting points.** The test step of Chapter 2 starts at the median labels. With twelve labels the
$\chi^2$ surface has more room for local minima. Start also from four other points (low and high [Fe/H] and
$T_{\rm eff}$, in scaled units) and keep, of the fits that converged, the one with the lowest $\chi^2$.

**Inside the training set?** Chapter 5 flagged extrapolation with N15's EFLAG. AnniesLasso tests whether the labels
lie inside the convex hull of the training labels, with a Delaunay triangulation; that does not work beyond a few
dimensions, but a small linear program does (`in_hull`).

**Saving a model** needs nothing but `np.savez`.

In [ ]:
t0 = time.time()
starts = np.zeros((5, len(NAMES)))
starts[1, 2], starts[2, 2], starts[3, 0], starts[4, 0] = -1.5, 1.0, -1.5, 1.5
lab_many, chi_many = label_set(ft, et, th12, s12, pv, sc, starts=starts)
lab_one, chi_one = label_set(ft, et, th12, s12, pv, sc)
dchi = chi_one - chi_many
better = dchi > 1
print(f'several starting points: chi^2 lower by > 1 for {better.sum()} of {len(ft)} test stars '
      f'(largest improvement {dchi.max():.0f}); their T_eff changes by up to {np.abs(lab_many - lab_one)[better, 0].max():.0f} K')
closer = np.abs(lab_many - Lt)[better] < np.abs(lab_one - Lt)[better]     # (stars, labels): closer to ASPCAP?
print(f'for these stars the new labels are closer to ASPCAP in {closer.mean():.0%} of the (star, label) pairs; '
      f'T_eff closer for {closer[:, 0].sum()} of {better.sum()}')
mp = labels[labels['sample'] == 'metalpoor'][NAMES[:3]].to_numpy()
inside = {'test stars, 3 labels': in_hull(Lt[:, :3], L[tr, :3]), 'test stars, 12 labels': in_hull(Lt, L[tr]),
          'metal-poor stars, 3 labels': in_hull(mp, L[tr, :3])}
for k, v in inside.items():
    print(f'inside the training-set hull: {v.mean():.2f} of {k}')
import tempfile
path = Path(tempfile.mkdtemp()) / 'model_12labels.npz'
np.savez(path, theta=th12, s=s12, pivot=pv, scale=sc, names=NAMES)
with np.load(path) as m:
    assert np.array_equal(m['theta'], th12) and list(m['names']) == NAMES
print(f'{time.time() - t0:.0f} s')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(lab_one[~better, 0], lab_many[~better, 0] - lab_one[~better, 0], '.', color='0.6', ms=3, label='same fit')
ax[0].plot(lab_one[better, 0], lab_many[better, 0] - lab_one[better, 0], 'o', color='C3', ms=5,
           label=r'lower $\chi^2$ from another start')
ax[0].set(xlabel=r'$T_{\rm eff}$ from the median start [K]', ylabel=r'$\Delta T_{\rm eff}$, best of 5 starts [K]',
          title='(a) several starting points')
ax[0].legend(fontsize=8)
ax[1].plot(L[tr, 0], L[tr, 1], '.', color='0.7', ms=2, label='training set')
ax[1].plot(mp[:, 0], mp[:, 1], '.', color='C3', ms=3, label='metal-poor sample (outside)')
ax[1].plot(Lt[inside['test stars, 12 labels'], 0], Lt[inside['test stars, 12 labels'], 1], '.', color='C0', ms=4,
           label='test stars inside the 12-label hull')
ax[1].invert_xaxis()
ax[1].invert_yaxis()
ax[1].set(xlabel=r'$T_{\rm eff}$ [K]', ylabel=r'$\log g$', title='(b) inside the training set?')
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Figure 10.5.** **(a)** Test stars labelled with the twelve-label model from the median labels only, and from the
best of five starting points: for most stars the two fits are the same (grey); for a few (red) another start finds
a lower $\chi^2$ and a different $T_{\rm eff}$. A lower $\chi^2$ is a better fit of the model, not necessarily
better labels: the printout above says how often the new labels are closer to ASPCAP's. **(b)** Kiel diagram of the training set (grey) and of the
metal-poor sample of Chapter 5 (red), all outside the convex hull of the training labels in ($T_{\rm eff}$,
$\log g$, [Fe/H]). In three dimensions almost every test star is inside the hull (fraction printed above). In
twelve dimensions only the blue ones are, fewer than one in five, although the test stars are disc giants drawn from
the same survey as the training stars: in many dimensions almost every new point lies outside the hull of a finite
sample. Being outside the twelve-dimensional hull therefore says little by itself (we have not measured whether
those stars have worse labels). Testing a few labels is more useful, at the price of missing extrapolation in the
others.

**Checks.**

In [ ]:
assert np.all(dchi > -1e-6) and better.sum() >= 1, 'more starting points must never give a worse fit'
assert inside['test stars, 3 labels'].mean() > 0.95 and inside['metal-poor stars, 3 labels'].mean() < 0.2
assert inside['test stars, 12 labels'].mean() < 0.5

## Summary

- Twelve labels make $T_{\rm eff}$ about a third more precise than three — but so does a cubic three-label model.
  Before attributing a gain to the extra labels, try a more flexible model with the same labels.
- L1 regularisation (C16) makes the model sparse and, when moderate, its predicted spectra better. In our tests it
  improved the labels only when the training set was not much larger than the number of coefficients (at 120
  stars, not at 300 or 1308; two random subsets per size, $\Lambda$ not re-tuned per size).
- Censoring makes each abundance a measurement from its own lines. At S/N 50–100 it improves Mg, Al, Si, Ca, Mn and
  Ni, and degrades O, C and $T_{\rm eff}$; the outcome depends on the element and on the windows (Holtzman et al.
  2018).
- Several starting points find a better fit for a few test stars in a hundred, some hundreds of kelvin away —
  but the new labels are closer to ASPCAP's for only about half of them. A convex-hull test of extrapolation is
  informative in three dimensions; in twelve it flags most ordinary stars.
- The appendix compares our regularised training step with AnniesLasso's.

## References

- Boyd, S., Parikh, N., Chu, E., Peleato, B., & Eckstein, J. 2011, *Distributed optimization and statistical learning
  via the alternating direction method of multipliers*, Foundations and Trends in Machine Learning 3, 1,
  [doi:10.1561/2200000016](https://doi.org/10.1561/2200000016) — ADMM for the lasso, §6.4.
- Casey, A. R., Hogg, D. W., Ness, M., et al. 2016, *The Cannon 2: A data-driven model of stellar spectra for
  detailed chemical abundance analyses*, [arXiv:1603.03040](https://ui.adsabs.harvard.edu/abs/2016arXiv160303040C)
  — L1-regularised 17-label model; choice of $\Lambda$ by validation.
- Casey, A. R., Hawkins, K., Hogg, D. W., et al. 2017, *The RAVE-on catalog of stellar atmospheric parameters and
  chemical abundances for chemo-dynamic studies in the Gaia era*,
  [ApJ 840, 59](https://ui.adsabs.harvard.edu/abs/2017ApJ...840...59C) — regularisation chosen by cross-validation.
- Holtzman, J. A., Hasselquist, S., Shetrone, M., et al. 2018, *APOGEE Data Releases 13 and 14: Data and
  Analysis*, [AJ 156, 125](https://ui.adsabs.harvard.edu/abs/2018AJ....156..125H) — the censored Cannon of DR14.
- Ness, M., Hogg, D. W., Rix, H.-W., Ho, A. Y. Q., & Zasowski, G. 2015, *The Cannon: a data-driven approach to
  stellar label determination*, [ApJ 808, 16](https://ui.adsabs.harvard.edu/abs/2015ApJ...808...16N).
- Tibshirani, R. 1996, *Regression shrinkage and selection via the lasso*, JRSS B 58, 267,
  [doi:10.1111/j.2517-6161.1996.tb02080.x](https://doi.org/10.1111/j.2517-6161.1996.tb02080.x).